# Check and download a chat model

See the model choices, check local files and required packages, and optionally download one. This notebook does not load a model or start a chat.

Run setup first. Select QAChat's `.venv` as the kernel, then run one cell at a time with Shift+Enter. Downloads start only if you change `DOWNLOAD` to `True`.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "app_core/models.py").is_file():
    root = root.parent
assert (root / "app_core/models.py").is_file(), "Open this notebook from QAChat."
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
assert Path(sys.prefix).resolve() == (root / ".venv").resolve(), "Select QAChat's .venv kernel."

from app_core.models import list_models, model_home, check_model, package_versions, download_model

print("Python:", sys.executable)

## 1. Choose where model files live

Keep `None` to use QAChat's ignored `models/` folder (or `QACHAT_MODEL_HOME` if you set it). To use your own folder, enter a path such as `Path("~/my-models")` instead.

This is the collection's root folder: each model lives inside `<model-name>/<revision>/`. Existing files in that layout are reused. Relative paths start at QAChat, even when this notebook starts in `notebooks/`.

After changing the folder, rerun this cell and the cells below it. Listing, checking, and downloading all use the same choice.

In [ ]:
MODEL_DIRECTORY = None
# For another folder: MODEL_DIRECTORY = Path("~/my-models")
print("Model storage:", model_home(MODEL_DIRECTORY))

## 2. See the available choices

This is our small supported file catalog, not an internet search. Listing and checking are offline. MLX models need an Apple Silicon Mac; PyTorch uses different weight files.

Sizes below are downloads on disk, not how much memory a running model will need.

In [ ]:
for model in list_models(directory=MODEL_DIRECTORY):
    print(f"{model['model']}: {model['backend']}, {model['size_gb']} GB, {model['status']}")

## 3. Choose a model and check its files

Use `qwen3-14b-mlx` on this Mac, or choose `qwen3.8-27b-mlx`. For the smaller PyTorch option, enter `qwen3-0.6b-torch`.

`missing` means no saved model; `incomplete` means some files are missing or have the wrong size. `downloaded` means all listed files have the expected sizes—it does not yet prove chat works.

In [ ]:
MODEL = "qwen3-14b-mlx"
report = check_model(MODEL, directory=MODEL_DIRECTORY)
report

## 4. Check the required packages

These packages live in `.venv`, separately from model files. A version means the package is installed; `MISSING` means you should rerun setup with that backend included. This reads package information, without loading a model. Use the setup notebook for a real backend calculation.

In [ ]:
for package, installed in package_versions(MODEL).items():
    print(f"{package}: {installed or 'MISSING'}")

## 5. Download only if you want to

Keep `False` to inspect without downloading. Change it to `True` to download the chosen model (see its size above). Correct files are reused, including local copies; incomplete downloads can be retried.

The notebook calls the same shared function the future app will use. It checks each file's content before accepting it. Use the notebook Stop button to cancel; partial files stay in the model folder and are not treated as complete.

In [ ]:
DOWNLOAD = False

if DOWNLOAD:
    saved_path = download_model(MODEL, directory=MODEL_DIRECTORY)
    print("Files checked and saved at:", saved_path)
else:
    print("Download skipped. No network access was needed.")

## 6. Check again

After a completed download, expect `downloaded` and empty missing/invalid file lists. For a slower check of every file's content, use `check_model(MODEL, directory=MODEL_DIRECTORY, verify=True)` and expect `verified`.

The reported folder contains weights, tokenizer files (which turn text into model inputs), and model settings. Download bookkeeping lives in its `.cache/huggingface` subfolder. The default `QAChat/models/` folder, including this bookkeeping, is excluded from Git. Packages stay in `.venv`, not in the model folder. If you choose a different folder inside the repository, add that folder to `.gitignore` too.

In [ ]:
report = check_model(MODEL, directory=MODEL_DIRECTORY)
report

Model preparation is the end of this example. Loading a model and sending a message will be the next notebook. Clear saved outputs before committing this notebook.